# FRASCI MFA D2 Reproduction

This notebook mirrors the known-good `Flow_MFA_D2_Best.ipynb` run: D1 diagnostic, D2 h1diag baseline, D2 h1diag best, and D2 balanced.


In [ ]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display

# Locate the FRASCI folder itself, regardless of where Jupyter was started.
CWD = Path.cwd().resolve()
FLOW_ROOT = None
for base in [CWD] + list(CWD.parents):
    if (base / "FRASCI" / "__init__.py").exists() and (base / "data" / "fcidump_cycle_6").exists():
        FLOW_ROOT = base
        break
    nested = base / "FRASCI"
    if (nested / "FRASCI" / "__init__.py").exists() and (nested / "data" / "fcidump_cycle_6").exists():
        FLOW_ROOT = nested
        break
if FLOW_ROOT is None:
    raise RuntimeError("Could not locate the FRASCI project folder")

PROJECT_ROOT = FLOW_ROOT.parent
if str(FLOW_ROOT) not in sys.path:
    sys.path.insert(0, str(FLOW_ROOT))

from FRASCI.mfa import run_mfa_d1, run_mfa_d2

FCIDUMP = FLOW_ROOT / "data" / "fcidump_cycle_6"
REF_DETS = FLOW_ROOT / "data" / "dets.npz"

# The known-good notebook used this diagonal gamma path. Keep a local copy here
# so the FRASCI folder can reproduce that workflow by itself.
GAMMA = FLOW_ROOT / "Outputs" / "meanfield_active" / "outs_extraction_autodets" / "gamma_mixed_final.npy"
if not GAMMA.exists():
    GAMMA = FLOW_ROOT / "Outputs" / "mfa" / "outs_extract_full_gamma_20260417_002006" / "gamma_mixed_diag.npy"

MFA_OUT = FLOW_ROOT / "Outputs" / "mfa"
REFERENCE_ENERGY = -327.1920
REFERENCE_DETS = 10_095
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"FLOW_ROOT    = {FLOW_ROOT}")
print(f"FCIDUMP      = {FCIDUMP}")
print(f"REF_DETS     = {REF_DETS}")
print(f"GAMMA        = {GAMMA}")
print(f"RUN_STAMP    = {RUN_STAMP}")

for required in (FCIDUMP, REF_DETS, GAMMA):
    if not required.exists():
        raise FileNotFoundError(required)


In [ ]:
def trimci_config(threshold, max_final_dets):
    return {
        "threshold": threshold,
        "max_final_dets": max_final_dets,
        "max_rounds": 2,
        "num_runs": 1,
        "pool_build_strategy": "heat_bath",
        "verbose": False,
    }


CONFIG_D1 = trimci_config(0.06, "auto")
CONFIG_D2_BASELINE = trimci_config(0.06, "auto")
CONFIG_D2_BEST = trimci_config(0.01, 1000)
CONFIG_D2_BALANCED = trimci_config(0.06, "auto")

display(Markdown("### selected-CI configurations"))
print("D1:", CONFIG_D1)
print("D2 baseline:", CONFIG_D2_BASELINE)
print("D2 best:", CONFIG_D2_BEST)
print("D2 balanced:", CONFIG_D2_BALANCED)


In [ ]:
d1_out = MFA_OUT / f"outs_notebook_d1_overlapping_{RUN_STAMP}"

d1 = run_mfa_d1(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D1,
    output_dir=str(d1_out),
)

display(Markdown(f"""
### D1 Result

- Output: `{d1_out}`
- fragment_n_dets: `{d1['fragment_n_dets']}`
- total_dets: `{d1['total_dets']}`
- matches Phase C baseline: `{d1['matches_phase_c_baseline']}`
- energy_status: `{d1['energy_status']}`
"""))


In [ ]:
d2_baseline_out = MFA_OUT / f"outs_notebook_d2_h1diag_threshold006_{RUN_STAMP}"

d2_baseline = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BASELINE,
    output_dir=str(d2_baseline_out),
    partition="h1diag",
)

display(Markdown(f"""
### D2 Baseline Result

- Output: `{d2_baseline_out}`
- E_total: `{d2_baseline['E_total']:.12f} Ha`
- error vs reference: `{d2_baseline['error_vs_reference']:+.12f} Ha`
- fragment_n_dets: `{d2_baseline['fragment_n_dets']}`
- total_dets: `{d2_baseline['total_dets']}`
- gamma_load_mode: `{d2_baseline['gamma_load_mode']}`
"""))


In [ ]:
d2_best_out = MFA_OUT / f"outs_notebook_d2_h1diag_threshold001_1000dets_{RUN_STAMP}"

d2_best = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BEST,
    output_dir=str(d2_best_out),
    partition="h1diag",
)

display(Markdown(f"""
### D2 Best Result

- Output: `{d2_best_out}`
- **E_total: `{d2_best['E_total']:.12f} Ha`**
- **error vs reference: `{d2_best['error_vs_reference']:+.12f} Ha`**
- fragment_n_dets: `{d2_best['fragment_n_dets']}`
- total_dets: `{d2_best['total_dets']}`
- E_total - DMET 1-shot B: `{d2_best['E_total_minus_dmet_1shot_b']:+.12f} Ha`
- gamma_load_mode: `{d2_best['gamma_load_mode']}`
"""))


In [ ]:
d2_balanced_out = MFA_OUT / f"outs_notebook_d2_balanced_threshold006_{RUN_STAMP}"

d2_balanced = run_mfa_d2(
    fcidump_path=str(FCIDUMP),
    gamma_path=str(GAMMA),
    ref_dets_path=str(REF_DETS),
    trimci_config=CONFIG_D2_BALANCED,
    output_dir=str(d2_balanced_out),
    partition="balanced",
)

display(Markdown(f"""
### D2 Balanced Result

- Output: `{d2_balanced_out}`
- E_total: `{d2_balanced['E_total']:.12f} Ha`
- error vs reference: `{d2_balanced['error_vs_reference']:+.12f} Ha`
- fragment_n_dets: `{d2_balanced['fragment_n_dets']}`
- total_dets: `{d2_balanced['total_dets']}`
- fragment_n_alpha: `{d2_balanced['fragment_n_alpha']}`
- fragment_n_beta: `{d2_balanced['fragment_n_beta']}`
"""))


In [ ]:
runs = [
    ("D1 overlapping", d1),
    ("D2 h1diag 0.06", d2_baseline),
    ("D2 h1diag 0.01 / 1000 dets", d2_best),
    ("D2 balanced 0.06", d2_balanced),
]

rows = []
for label, result in runs:
    rows.append({
        "run": label,
        "partition": result.get("partition"),
        "E_total": result.get("E_total"),
        "error_vs_reference": result.get("error_vs_reference"),
        "fragment_n_dets": result.get("fragment_n_dets"),
        "total_dets": result.get("total_dets"),
        "gamma_load_mode": result.get("gamma_load_mode"),
    })

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except Exception:
    for row in rows:
        print(row)

best_energy = d2_best["E_total"]
best_error = d2_best["error_vs_reference"]
display(Markdown(f"""
### Headline

The best regenerated D2 result in this notebook is:

```text
E_total = {best_energy:.12f} Ha
error vs {REFERENCE_ENERGY:.4f} Ha = {best_error:+.12f} Ha
fragment_n_dets = {d2_best['fragment_n_dets']}
total_dets = {d2_best['total_dets']}
```
"""))


In [ ]:
display(Markdown(f"""
### Output folders created by this notebook

- D1: `{d1_out}`
- D2 baseline: `{d2_baseline_out}`
- D2 best: `{d2_best_out}`
- D2 balanced: `{d2_balanced_out}`
"""))
